In [1]:
%run 00_utils.ipynb

%load_ext autoreload
%autoreload 2

Root-Verzeichnis: /home/jessh/Study/Bachelor-Thesis
Datensätze: Anti-UAV410 + CST-AntiUAV


In [2]:
from pathlib import Path
import cv2, os
import numpy as np
from data_handler.dataset import AntiUAVDataset

import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider

In [3]:
# Trainings-Frames aller Datensätze in DATASETS (Reihenfolge wie in DATASETS).
# Nur einer: AntiUAVDataset(DatasetName.CST_ANTI_UAV, split="train", debug=True)
train_data = AntiUAVDataset(DATASETS, split="train", debug=True)

# Index-Bereich je Datensatz (für IMAGE_ID / train_data[...])
_start = 0
for ds in DATASETS:
    n = sum(r.dataset is ds for r in train_data.samples)
    print(f"{ds.value:<14} Index {_start} .. {_start + n - 1}")
    _start += n

25-11-2026 03:08:36 | INFO    | [train] Anti-UAV410: 213995 frames in 200 sequences (2.4 s)
25-11-2026 03:08:38 | INFO    | [train] CST-AntiUAV: 127991 frames in 120 sequences (1.6 s)


Anti-UAV410    Index 0 .. 213994
CST-AntiUAV    Index 213995 .. 341985


In [4]:
def visualise(img_org, img_aug):
    _, ax = plt.subplots(1, 3, figsize=(15, 5))
    vmin, vmax = img_org.min(), img_org.max()   # gleicher Skalenbereich für faire Vergleichbarkeit
    
    ax[0].imshow(img_org, cmap='gray', vmin=vmin, vmax=vmax); ax[0].set_title(f"original")
    ax[1].imshow(img_aug, cmap='gray', vmin=vmin, vmax=vmax); ax[1].set_title(f"augmented")
    ax[2].imshow(img_aug.astype(np.float32) - img_org.astype(np.float32), cmap='seismic');          ax[2].set_title("difference")
    
    for a in ax: 
        a.axis('off')
        
    plt.tight_layout()
    plt.show()

In [5]:
img, target, img_org = train_data[100000]

In [6]:
from augmentations.ir import netd_noise

# NETD-ähnliches Rauschen: T_noise = T_ref + eta, eta ~ N(0, sigma^2), sigma relativ zu [0, 1]
def explore(sigma):
    out = netd_noise(img_org, sigma=sigma)
    visualise(img_org, out)

interact(
    explore,
    sigma = FloatSlider(min=0.0, max=0.05, step=0.0005, value=0.0, readout_format='.4f',
                        continuous_update=False, description='sigma'),
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='sigma', max=0.05, readout_f…

In [7]:
from augmentations.ir import blurring_with_psf

def explore(sigma_0=0.3, rho=1.0, theta_deg=0):
    out = blurring_with_psf(
        img_org, sigma_0=sigma_0, rho=rho,
        theta=np.radians(theta_deg)
    )
    visualise(img_org, out)

WIDE = {'description_width': 'initial'}   # volle Beschriftung anzeigen

interact(
    explore,
    sigma_0   = FloatSlider(min=0.1,  max=4.0, step=0.1,  value=0.1, continuous_update=False, style=WIDE,
                            description='blur strength'),
    rho       = FloatSlider(min=0.15, max=1.0, step=0.05, value=1.0, continuous_update=False, style=WIDE,
                            description='stretching'),
    theta_deg = FloatSlider(min=0,    max=180, step=5,    value=0,   continuous_update=False, style=WIDE,
                            description='direction'),
);

interactive(children=(FloatSlider(value=0.1, continuous_update=False, description='blur strength', max=4.0, mi…

In [8]:
from augmentations.ir import fpn

def explore(sigma_gain=0.01, sigma_offset=2.0, mode="pixel"):
    out = fpn(
        img_org, 
        sigma_gain=sigma_gain, 
        sigma_offset=sigma_offset, 
        mode=mode
    )
    
    visualise(img_org, out)

# Interaktiver Widget Explorer für das neue FPN
interact(
    explore,
    sigma_gain   = FloatSlider(min=0.0, max=0.5, step=0.025,  value=0.0, readout_format='.3f', continuous_update=False),
    sigma_offset = FloatSlider(min=0.0, max=10.0, step=0.5,   value=0.0, readout_format='.1f', continuous_update=False),
    mode         = widgets.Dropdown(options=["pixel", "column", "row"], value="pixel", description="mode")
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='sigma_gain', max=0.5, reado…

In [9]:
from augmentations.ir import thermal_histogram_shift

def explore(delta_limit=30):
    out = thermal_histogram_shift(
        img_org, 
        delta_limit
    )

    visualise(img_org, out)

interact(
    explore,
    delta_limit   = FloatSlider(min=0.0, max=100,   step=5,   value=0.0, readout_format='.2f', continuous_update=False),
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='delta_limit', step=5.0), Ou…

In [10]:
from augmentations.ir import salt_and_peper

def explore(p, q):
    out = salt_and_peper(
        img_org, 
        p=p, q=q
    )

    visualise(img_org, out)

WIDE = {'description_width': 'initial'}   # volle Beschriftung anzeigen

interact(
    explore,
    p = FloatSlider(min=0.0, max=0.12, step=0.005, value=0.0, readout_format='.3f', continuous_update=True, style=WIDE,
                    description='Black pixel'),
    q = FloatSlider(min=0.0, max=0.12, step=0.005, value=0.0, readout_format='.3f', continuous_update=True, style=WIDE,
                    description='White pixel'),
);

interactive(children=(FloatSlider(value=0.0, description='Black pixel', max=0.12, readout_format='.3f', step=0…

## ACB (Tong et al. 2024)

`ACB` (`src/augmentations/sota.py`) auf dem Frame `ACB_IMAGE_ID`: kleine Trainingsziele (Boxfläche < `max area ratio`
der Bildfläche) mit Umgebung kopieren (Bounding-box copy), mit RandAugment verändern (`num_ops` Operationen aus `ops`,
Stärke `magnitude` 0–10) und adaptiv einfügen (I' = T'·T + (1 − T')·I; `fusion`: `paper` T' = T / max T wie im Paper,
`background` T' = max(T − b, 0) / max(T − b) mit b = Median um die Box (nicht im Paper), `none` einfach kopieren).
GT-Box grün, eingefügte Boxen rot, darunter ein Zoom auf jedes eingefügte Ziel (Box ± 20 px). `seed` wählt andere
Ziele/Positionen. Abweichungen vom Paper: `src/augmentations/ACB_deviations.md`.

In [11]:
import albumentations as A
from ipywidgets import IntSlider, Dropdown, SelectMultiple
from augmentations.sota import ACB, OPS, DEFAULT_OPS, FUSIONS

ACB_IMAGE_ID = 100000
acb_sample = train_data.samples[ACB_IMAGE_ID]
_, _, acb_img = train_data[ACB_IMAGE_ID]                            # Original, Graustufen uint8
acb_gt = [acb_sample.bbox] if acb_sample.bbox is not None else []   # COCO: (x, y, w, h)


def explore(seed=0, max_area_ratio_pct=0.15, num_targets=1, num_ops=2, magnitude=5.0, ops=DEFAULT_OPS, context=1.0,
            min_context_px=8, fusion="paper"):
    if num_ops > 0 and not ops:
        print("ops: mindestens eine Operation wählen")
        return
    # Quellen: nur Boxen mit Fläche / Bildfläche < max_area_ratio (kleine Ziele, wie im Paper)
    t = ACB(train_data.samples, max_area_ratio=round(max_area_ratio_pct / 100, 6), num_targets=num_targets,
            num_ops=num_ops, magnitude=magnitude, ops=list(ops), context=context, min_context_px=min_context_px,
            fusion=fusion, p=1.0)
    comp = A.Compose([t], bbox_params=A.BboxParams(format="coco", label_fields=["class_labels"]))
    comp.set_random_seed(seed)
    out = comp(image=acb_img, bboxes=acb_gt, class_labels=[0] * len(acb_gt))
    img, boxes = out["image"], [tuple(b) for b in out["bboxes"]]
    gt, pasted = boxes[:len(acb_gt)], boxes[len(acb_gt):]   # ACB hängt die neuen Boxen hinten an

    print(f"{len(t.sources)} Quellziele, {len(pasted)} von {num_targets} Zielen eingefügt")
    fig, ax = plt.subplots(1, 2, figsize=(16, 6.5))
    for a, im, title in ((ax[0], acb_img, "original"), (ax[1], img, "ACB")):
        a.imshow(im, cmap="gray", vmin=0, vmax=255)
        a.set_title(title)
        a.axis("off")
    for x, y, w, h in gt:
        for a in ax:
            a.add_patch(plt.Rectangle((x - 2.5, y - 2.5), w + 4, h + 4, fill=False, ec="lime", lw=1))
    for x, y, w, h in pasted:
        ax[1].add_patch(plt.Rectangle((x - 2.5, y - 2.5), w + 4, h + 4, fill=False, ec="red", lw=1))
    plt.tight_layout()
    plt.show()

    if pasted:
        fig, axes = plt.subplots(1, len(pasted), figsize=(3 * len(pasted), 3), squeeze=False)
        for a, (x, y, w, h) in zip(axes[0], pasted):
            x0, y0 = max(int(x - 20), 0), max(int(y - 20), 0)
            a.imshow(img[y0:int(y + h + 20), x0:int(x + w + 20)], cmap="gray", vmin=0, vmax=255,
                     interpolation="nearest")
            a.set_title(f"{w:.0f}×{h:.0f} px", fontsize=9)
            a.axis("off")
        plt.show()


WIDE = {'description_width': 'initial'}   # volle Beschriftung anzeigen

interact(
    explore,
    seed               = IntSlider(min=0, max=200, step=1, value=0, continuous_update=False, style=WIDE),
    max_area_ratio_pct = FloatSlider(min=0.05, max=5.0, step=0.05, value=0.15, readout_format='.2f',
                                     continuous_update=False, style=WIDE, description='max area ratio (%)'),
    num_targets        = IntSlider(min=0, max=6, step=1, value=1, continuous_update=False, style=WIDE,
                                   description='targets per image'),
    num_ops            = IntSlider(min=0, max=6, step=1, value=2, continuous_update=False, style=WIDE,
                                   description='num_ops (N)'),
    magnitude          = FloatSlider(min=0.0, max=10.0, step=0.5, value=5.0, continuous_update=False, style=WIDE,
                                     description='magnitude (M)'),
    ops                = SelectMultiple(options=sorted(OPS), value=tuple(DEFAULT_OPS), rows=8, description='ops'),
    context            = FloatSlider(min=0.0, max=3.0, step=0.1, value=1.0, continuous_update=False, style=WIDE,
                                     description='context (x box)'),
    min_context_px     = IntSlider(min=0, max=30, step=1, value=8, continuous_update=False, style=WIDE,
                                   description='min context (px)'),
    fusion             = Dropdown(options=list(FUSIONS), value="paper", description='fusion'),
);

interactive(children=(IntSlider(value=0, continuous_update=False, description='seed', max=200, style=SliderSty…

## PGDAA (PGDA adaptiert)

Synthetisches Bild von `PGDAA` (`src/augmentations/sota.py`): Hintergrund ohne Ziel (Train-Frames aus `DATASETS`),
Subfigure-Split, eingefügte Bibliotheksziele (rote Boxen) und darunter ein Zoom auf jedes Ziel (Box ± 20 px).
`seed` wählt ein anderes Bild. `max_area_ratio` und `scr_percentiles` ändern die Prioren A und S; dafür wird der
Transform neu gebaut (ca. 2 s, danach gecacht). Abweichungen vom Paper: `src/augmentations/PGDAA_deviations.md`.
Voraussetzung: Bibliothek gebaut (`python src/main.py -l`).

In [12]:
import functools
import albumentations as A
from ipywidgets import IntSlider, Checkbox, FloatRangeSlider, IntRangeSlider
from augmentations.sota import PGDAA

pgdaa_records = train_data.samples  # Train-Frames von DATASETS: Hintergründe B_O und Prior A


@functools.lru_cache(maxsize=4)
def pgdaa_for(max_area_ratio, scr_lo, scr_hi):
    # A und S entstehen beim Erzeugen; alle anderen Parameter werden pro Aufruf gesetzt
    return PGDAA(pgdaa_records, max_area_ratio=max_area_ratio, scr_percentiles=(scr_lo, scr_hi), p=1.0)


def explore(seed=0, num_targets=2, subfigure_split=True, crop_scale=(0.5, 1.0), max_area_ratio_pct=0.15,
            area_jitter=0.1, scr_perception=True, scr_percentiles=(20, 80), gaussian_sigma=1.0,
            brightness=0.1, contrast=0.1):
    if scr_percentiles[0] >= scr_percentiles[1]:
        print("scr_percentiles: unterer Wert muss kleiner als der obere sein")
        return
    t = pgdaa_for(round(max_area_ratio_pct / 100, 6), *scr_percentiles)
    t.num_targets = (num_targets, num_targets)
    t.subfigure_split = subfigure_split
    t.crop_scale = tuple(crop_scale)
    t.area_jitter = area_jitter
    t.scr_perception = scr_perception
    t.gaussian_sigma = gaussian_sigma
    t.color_jitter = {"brightness": brightness, "contrast": contrast}

    comp = A.Compose([t], bbox_params=A.BboxParams(format="coco", label_fields=["class_labels"]))
    comp.set_random_seed(seed)
    out = comp(image=np.zeros((512, 640), np.uint8), bboxes=[], class_labels=[])  # Bild wird ersetzt
    img, boxes = out["image"], [tuple(b) for b in out["bboxes"]]

    print(f"A: {len(t.A)} Werte < {max_area_ratio_pct:g} %   S: {len(t.S)} Werte in [{t.S.min():.2f}, {t.S.max():.2f}]   "
          f"eingefügt: {len(boxes)}")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    for x, y, w, h in boxes:
        ax.add_patch(plt.Rectangle((x - 2.5, y - 2.5), w + 4, h + 4, fill=False, ec="red", lw=1))
    ax.axis("off")
    plt.show()

    if boxes:
        fig, axes = plt.subplots(1, len(boxes), figsize=(3 * len(boxes), 3), squeeze=False)
        for ax, (x, y, w, h) in zip(axes[0], boxes):
            x0, y0 = max(int(x - 20), 0), max(int(y - 20), 0)
            ax.imshow(img[y0:int(y + h + 20), x0:int(x + w + 20)], cmap="gray", vmin=0, vmax=255,
                      interpolation="nearest")
            ax.set_title(f"{w:.0f}×{h:.0f} px", fontsize=9)
            ax.axis("off")
        plt.show()


WIDE = {'description_width': 'initial'}   # volle Beschriftung anzeigen

interact(
    explore,
    seed               = IntSlider(min=0, max=200, step=1, value=0, continuous_update=False, style=WIDE),
    num_targets        = IntSlider(min=0, max=6, step=1, value=1, continuous_update=False, style=WIDE,
                                   description='targets per image'),
    subfigure_split    = Checkbox(value=True, description='subfigure split'),
    crop_scale         = FloatRangeSlider(min=0.1, max=1.0, step=0.05, value=(0.5, 1.0), continuous_update=False,
                                          style=WIDE, description='crop scale'),
    max_area_ratio_pct = FloatSlider(min=0.05, max=1.0, step=0.05, value=0.15, readout_format='.2f',
                                     continuous_update=False, style=WIDE, description='max area ratio (%)'),
    area_jitter        = FloatSlider(min=0.0, max=0.5, step=0.05, value=0.1, continuous_update=False, style=WIDE,
                                     description='area jitter'),
    scr_perception     = Checkbox(value=True, description='SCR perception'),
    scr_percentiles    = IntRangeSlider(min=0, max=100, step=5, value=(20, 80), continuous_update=False, style=WIDE,
                                        description='SCR percentiles'),
    gaussian_sigma     = FloatSlider(min=0.0, max=4.0, step=0.25, value=1.0, continuous_update=False, style=WIDE,
                                     description='Gaussian paste sigma'),
    brightness         = FloatSlider(min=0.0, max=0.5, step=0.05, value=0.1, continuous_update=False, style=WIDE,
                                     description='jitter brightness'),
    contrast           = FloatSlider(min=0.0, max=0.5, step=0.05, value=0.1, continuous_update=False, style=WIDE,
                                     description='jitter contrast'),
);

interactive(children=(IntSlider(value=0, continuous_update=False, description='seed', max=200, style=SliderSty…